# YOLO Training Workflow

This notebook is a clean template for training, validating, and testing detection models.
It is organized to minimize repeated code and make experiments easier to track.


## 1. Environment Setup

Run this cell first to load dependencies, detect the project root, and verify CUDA.


In [1]:
from __future__ import annotations

import gc
from datetime import date
from pathlib import Path

import torch
import ultralytics
from ultralytics import YOLO
from IPython.display import Image, display

# Resolve project root (works if notebook is run from project root or notebooks/)
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")

RUN_ULTRALYTICS_CHECKS = False  # Set True only when you want a full environment check
if RUN_ULTRALYTICS_CHECKS:
    ultralytics.checks()


Project root: /home/robiotec/Documents/Entrenamientos/Training
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Torch version: 2.10.0+cu128


## 2. Shared Paths and Defaults

Keep all reusable paths and defaults in one place.


In [ ]:
BASE_MODEL_PATH = PROJECT_ROOT / "model" / "yolo26n.pt"
RESULT_ROOT = PROJECT_ROOT / "result"

YAML_CONFIGS = {
    "caja": PROJECT_ROOT / "configs" / "caja.yaml",
    "vetas": PROJECT_ROOT / "configs" / "vetas.yaml",
    "mixed": PROJECT_ROOT / "configs" / "mixto.yaml",
}

# Shared defaults for training. Override per experiment only when needed.
DEFAULT_TRAIN_ARGS = {
    "epochs": 150,
    "imgsz": 640,
    "device": 0,
    "warmup_epochs": 3,
    "seed": 42,
    "lrf": 0.1,
    "weight_decay": 0.0005,
    "workers": 8,
    "cache": True,
    "plots": True,
}

print(f"Base model: {BASE_MODEL_PATH}")
for name, cfg in YAML_CONFIGS.items():
    print(f"{name:>5}: {cfg}")


## 3. Utility Functions

These helpers remove duplicated code for CUDA cleanup, train, validate, and predict.


In [ ]:
def clean_cuda(verbose: bool = True) -> None:
    # Release cached GPU memory after heavy operations.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            allocated = torch.cuda.memory_allocated() / 1024**2
            reserved = torch.cuda.memory_reserved() / 1024**2
            print(f"GPU memory - allocated: {allocated:.2f} MB | reserved: {reserved:.2f} MB")


def train_model(
    run_group: str,
    run_name: str,
    data_yaml: Path,
    model_path: Path = BASE_MODEL_PATH,
    **overrides,
):
    # Train a model and store outputs under result/<run_group>/<run_name>.
    args = dict(DEFAULT_TRAIN_ARGS)
    args.update(overrides)

    model = YOLO(str(model_path))
    output = model.train(
        data=str(data_yaml),
        project=str(RESULT_ROOT / run_group),
        name=run_name,
        **args,
    )
    clean_cuda()
    return output


def validate_model(
    model_path: Path,
    data_yaml: Path,
    run_group: str,
    run_name: str,
    split: str = "test",
    conf: float = 0.25,
    **overrides,
):
    # Run validation and save metrics under result/<run_group>/<run_name>.
    model = YOLO(str(model_path))
    output = model.val(
        data=str(data_yaml),
        split=split,
        conf=conf,
        project=str(RESULT_ROOT / run_group),
        name=run_name,
        save_json=True,
        plots=True,
        **overrides,
    )
    clean_cuda()
    return output


def predict_images(
    model_path: Path,
    source_path: Path,
    run_group: str,
    run_name: str,
    conf: float = 0.25,
    **overrides,
):
    # Run inference on images and save visual predictions.
    model = YOLO(str(model_path))
    output = model.predict(
        source=str(source_path),
        conf=conf,
        project=str(RESULT_ROOT / run_group),
        name=run_name,
        save=True,
        **overrides,
    )
    clean_cuda()
    return output


## 4. Quick GPU Cleanup

Run this anytime you want to free cached CUDA memory with one click.


In [ ]:
clean_cuda()

## 5. Configure One Training Run

Edit only this cell for each experiment.


In [ ]:
today = date.today().isoformat()

RUN_CONFIG = {
    "target": "Caja",  # Use "Caja", "Vetas", or "mixed"
    "yaml": "caja",  # Use "caja", "vetas", or "mixed"
    "split_tag": "split_name_here",  # Example: "2026-03-06_caja_clean_v1"
    "run_id": "run_001",
    "train_args": {
        "batch": 64,
        "patience": 30,
        "lr0": 0.01,
    },
}

RUN_NAME = f"{RUN_CONFIG['split_tag']}__{RUN_CONFIG['run_id']}"
DATA_YAML = YAML_CONFIGS[RUN_CONFIG["yaml"]]
RUN_GROUP = RUN_CONFIG["target"]

print(f"Training group: {RUN_GROUP}")
print(f"Run name: {RUN_NAME}")
print(f"YAML config: {DATA_YAML}")


## 6. Train


In [ ]:
train_results = train_model(
    run_group=RUN_GROUP,
    run_name=RUN_NAME,
    data_yaml=DATA_YAML,
    **RUN_CONFIG["train_args"],
)


## 7. Validate on Test Split

Set `MODEL_TO_VALIDATE` to your best checkpoint.


In [ ]:
MODEL_TO_VALIDATE = RESULT_ROOT / RUN_GROUP / RUN_NAME / "weights" / "best.pt"

val_results = validate_model(
    model_path=MODEL_TO_VALIDATE,
    data_yaml=DATA_YAML,
    run_group=RUN_GROUP,
    run_name=f"{RUN_NAME}__val_test",
    split="test",
    conf=0.316,
)


## 8. Predict on External Images (Optional)

Set `PREDICT_SOURCE` to any folder of images.


In [ ]:
PREDICT_SOURCE = Path("/path/to/images")

# Uncomment to run prediction
# pred_results = predict_images(
#     model_path=MODEL_TO_VALIDATE,
#     source_path=PREDICT_SOURCE,
#     run_group=RUN_GROUP,
#     run_name=f"{RUN_NAME}__predict",
#     conf=0.5,
# )


## 9. Quick Visual Comparison (Optional)


In [ ]:
# Update image paths if you want to compare confusion matrices side by side.
# from IPython.display import Image, display
#
# img1 = Image(filename=str(RESULT_ROOT / "Caja" / "some_run" / "confusion_matrix_normalized.png"), width=450)
# img2 = Image(filename=str(RESULT_ROOT / "Caja" / "some_run__val_test" / "confusion_matrix_normalized.png"), width=450)
#
# try:
#     from ipywidgets import HBox
#     display(HBox([img1, img2]))
# except Exception:
#     # Fallback without ipywidgets
#     display(img1)
#     display(img2)
